# Distributed Portfolio Optimization via Divide-and-Conquer QAOA on CUDA-Q

This notebook walks through an end-to-end pipeline that uses the
**Quantum Approximate Optimization Algorithm (QAOA)** — distributed across
multiple GPUs via NVIDIA's CUDA-Q framework — to solve a real
**Markowitz portfolio optimization** problem on S&P 500 stocks.

## What you will learn

1. **Data pipeline** — fetching real stock data and computing the financial
   inputs Markowitz needs (expected returns, covariance, correlation).
2. **QUBO formulation** — turning a constrained optimization problem into
   a *Quadratic Unconstrained Binary Optimization* form a quantum computer
   can solve.
3. **Ising mapping** — translating the QUBO into a Hamiltonian whose
   ground state is the optimal portfolio.
4. **Stock clustering** — using hierarchical clustering on the correlation
   matrix to split a 15-qubit problem into smaller, parallelizable
   subproblems.
5. **QAOA** — the quantum algorithm itself: how the circuit is built,
   what its parameters mean, and how a classical optimizer drives it.
6. **Distributed execution** — running the per-cluster QAOA in parallel
   across 4 GPUs using CUDA-Q's MQPU backend.
7. **Merging** — recombining the sub-portfolios into a global solution
   and refining it with greedy local search.

## Prerequisites

- **Quantum**: qubits, single-qubit gates (H, Rx, Rz), two-qubit gates
  (CNOT), and the idea of measuring in the Z basis.
- **Python**: numpy, basic plotting.

You do **not** need a finance background — Section 2 covers the relevant
concepts from scratch. Run this notebook from the project root so that
`from src.X import ...` resolves.


## 2. The Portfolio Optimization Problem

Suppose you have a universe of $N$ stocks and a budget to pick exactly $B$
of them. You want to maximise expected return while minimising risk. This
is the classical **Markowitz mean-variance** problem.

Let $x \in \{0, 1\}^N$ be a binary selection vector ($x_i = 1$ means we
hold stock $i$, otherwise $0$). Let

- $\mu \in \mathbb{R}^N$ — vector of (annualised) expected returns,
- $\Sigma \in \mathbb{R}^{N \times N}$ — covariance matrix of returns.

The portfolio's expected return is $\mu^\top x$ and its variance is
$x^\top \Sigma x$. Markowitz balances the two with a risk-aversion
parameter $q \ge 0$:

$$
\min_{x \in \{0,1\}^N} \;\; -\mu^\top x + q \, x^\top \Sigma x
\qquad \text{subject to} \qquad \sum_i x_i = B.
$$

When $q$ is small we focus on returns; when $q$ is large we focus on
diversification. The budget constraint forces us to pick exactly $B$
assets — without it, the optimum is the empty portfolio (no risk).

References for this section: **[1]** Markowitz (1952), **[2]** Buonaiuto
et al. (2023). Full details in Section 11.


## 3. Data Pipeline

We use **15 stocks** drawn from 5 sectors (3 tickers each), giving the
clustering step something interesting to find while keeping the qubit
count manageable:

| Sector              | Tickers                |
|---------------------|------------------------|
| Tech                | AAPL, MSFT, GOOGL      |
| Finance             | JPM, GS, BAC           |
| Energy              | XOM, CVX, COP          |
| Healthcare          | JNJ, PFE, UNH          |
| Consumer staples    | PG, KO, WMT            |

The window is **2022-01-01 to 2024-12-31** — three years, covering both
the 2022 bear market and the 2023–2024 rally, so the covariance estimate
isn't dominated by a single regime.

First, set up the path and import core scientific Python.


In [ ]:
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)


Fetch daily adjusted closing prices for the 15 tickers via `yfinance`.
This makes a network call.


In [ ]:
from src.data_pipeline import (
    fetch_stock_data, compute_log_returns, compute_financial_metrics,
    TICKERS, START_DATE, END_DATE,
)

print(f"Fetching {len(TICKERS)} tickers from {START_DATE} to {END_DATE}...")
prices = fetch_stock_data(TICKERS, START_DATE, END_DATE)
print(f"Prices shape: {prices.shape}")
prices.head()


### Why log returns instead of simple returns?

Given a price series $P_t$, two natural definitions of one-period return
are:

- **Simple return:** $r_t = (P_t - P_{t-1}) / P_{t-1}$.
- **Log return:** $\ell_t = \ln(P_t / P_{t-1})$.

Quants almost always use log returns, for four reasons:

1. **Time-additivity.** Multi-day log return $\ln(P_{t+k}/P_t)$ equals the
   sum $\ell_{t+1} + \ell_{t+2} + \cdots + \ell_{t+k}$. Simple returns
   compound (multiply), they don't add — so you can't average them.
2. **Symmetry under reversal.** A $+10\%$ simple gain followed by a
   $-10\%$ simple loss leaves you at $0.99$, not $1.00$. Log returns are
   well-behaved under sign-flips and unbiased.
3. **Closer to Gaussian.** Markowitz implicitly assumes returns are
   roughly normally distributed. Log returns fit that assumption far
   better than simple returns (which can't go below $-1$).
4. **Indistinguishable for small moves.** For daily moves of a few
   percent, $\ln(1+r) \approx r$, so we lose nothing in practice.

Compute log returns from the price matrix.


In [ ]:
returns = compute_log_returns(prices)
print(f"Log returns shape: {returns.shape}")
print(f"Sample daily values:")
returns.head()


### Annualization

Daily mean and covariance are tiny numbers. To get human-readable annual
figures we **scale by trading days per year**:

- $\hat{\mu}_{\text{annual}} = 252 \cdot \hat{\mu}_{\text{daily}}$
- $\hat{\Sigma}_{\text{annual}} = 252 \cdot \hat{\Sigma}_{\text{daily}}$

Why $252$? Markets are open on weekdays minus US holidays — about 252
trading days per year on average.

Why does **covariance** scale by $252$ and not $252^2$? If daily returns
are independent and identically distributed, the variance of a sum of
$N$ daily returns is $N$ times the daily variance:

$$
\operatorname{Var}\!\left(\textstyle \sum_{t=1}^{N} \ell_t\right)
\;=\; N \cdot \operatorname{Var}(\ell_1).
$$

So variance scales linearly in time, and so does covariance. Volatility
(standard deviation) scales as $\sqrt{N}$ — that's why annual volatility
is roughly $16\times$ daily volatility ($\sqrt{252} \approx 15.87$).

Compute the three Markowitz inputs $\mu$, $\Sigma$, $\rho$.


In [ ]:
mu, sigma, rho = compute_financial_metrics(returns)

print("Annualized expected returns mu:")
for t, m in zip(TICKERS, mu):
    print(f"  {t:>5s}  {m:+.4f}")
print(f"\nCovariance matrix sigma : {sigma.shape}")
print(f"Correlation matrix rho  : {rho.shape}")


The correlation matrix $\rho$ is the most informative single object —
it controls the clustering downstream. Plot it as a heatmap.


In [ ]:
import seaborn as sns
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(rho, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            xticklabels=TICKERS, yticklabels=TICKERS, ax=ax,
            cbar_kws={"label": "correlation"})
ax.set_title("Stock return correlation matrix (2022–2024)")
plt.tight_layout()
plt.show()


### Reading the heatmap

Look for the dark red blocks along the diagonal — those are sectors:

- **Energy** (XOM, CVX, COP): correlations around $0.83$ — they all move
  with oil prices.
- **Finance** (JPM, GS, BAC): around $0.77$ — they all move with rates
  and credit.
- **Tech** (AAPL, MSFT, GOOGL): around $0.67$ — looser block, since these
  are also exposed to broader market beta.

This block structure is the entire reason clustering will work. If every
stock were equally correlated with every other, splitting them into groups
would lose nearly all the information.


## 4. QUBO Formulation

A quantum optimisation device doesn't understand "minimise $f(x)$ subject
to $g(x) = 0$". It finds the ground state of a Hamiltonian. So we have
two jobs:

1. Fold the **budget constraint** into the objective as a penalty term.
2. Write the result as $x^\top Q x$ for some matrix $Q$ — the
   **QUBO** form.

### Step 1: penalising the constraint

We want $\sum_i x_i = B$. Define a penalty:

$$
P(x) \;=\; \lambda \left( \sum_i x_i - B \right)^2.
$$

If exactly $B$ stocks are selected, $P(x) = 0$. Otherwise $P(x) > 0$.
Pick $\lambda$ large enough that breaking the constraint is always worse
than any feasible portfolio.

Expanding:

$$
\left(\sum_i x_i - B\right)^2 \;=\; \sum_i x_i^2 + 2 \sum_{i < j} x_i x_j - 2 B \sum_i x_i + B^2.
$$

Since $x_i \in \{0,1\}$ implies $x_i^2 = x_i$, this simplifies to

$$
P(x) \;=\; \lambda \left[ (1 - 2B) \sum_i x_i + 2 \sum_{i<j} x_i x_j + B^2 \right].
$$

The constant $\lambda B^2$ doesn't affect the argmin — it ends up in an
*offset*.

### Step 2: combining into a single matrix Q

Adding the Markowitz objective $-\mu^\top x + q\, x^\top \Sigma x$ to the
penalty and collecting $x_i x_j$ terms:

$$
\boxed{ \quad
Q_{ii}    = -\mu_i + q\,\Sigma_{ii} + \lambda(1 - 2B), \qquad
Q_{ij}    = q\,\Sigma_{ij} + \lambda \;\;\;(i \ne j).
\quad}
$$

Then $\text{cost}(x) = x^\top Q x + \text{const}$, and we minimise
$x^\top Q x$ over $\{0,1\}^N$.

Reference: **[2]** Buonaiuto et al. (2023), eq. 7–11.

Now build $Q$ for the 15-stock universe. We use $B = 6$, $q = 0.5$,
$\lambda = 10$ — the penalty is large enough to dominate any reasonable
Markowitz term so that infeasible portfolios get crushed.


In [ ]:
from src.qubo import build_qubo, brute_force_qubo

BUDGET = 6
Q_RISK = 0.5
PENALTY = 10.0

Q = build_qubo(mu, sigma, q=Q_RISK, budget=BUDGET, penalty=PENALTY)
print(f"Q shape       : {Q.shape}")
print(f"Q symmetric   : {np.allclose(Q, Q.T)}")
print(f"Q diagonal Q_ii (first 5): {np.diag(Q)[:5]}")


Sanity-check the QUBO on a 4-stock toy problem. With only 4 binary
variables there are $2^4 = 16$ candidate portfolios — small enough to
brute-force every one and confirm the QUBO actually encodes the
optimisation we want.


In [ ]:
toy_idx = [0, 5, 6, 11]   # AAPL, BAC, XOM, UNH (cross-sector picks)
toy_mu = mu[toy_idx]
toy_sigma = sigma[np.ix_(toy_idx, toy_idx)]
toy_tickers = [TICKERS[i] for i in toy_idx]

Q_toy = build_qubo(toy_mu, toy_sigma, q=Q_RISK, budget=2, penalty=10.0)
x_opt, val_opt = brute_force_qubo(Q_toy, budget=2)
print(f"Toy universe: {toy_tickers}, budget = 2")
print(f"Optimal x         : {x_opt.astype(int)}")
print(f"  → selects        : {[t for t,b in zip(toy_tickers, x_opt) if b==1]}")
print(f"QUBO objective    : {val_opt:.4f}")
print(f"Expected return   : {float(toy_mu @ x_opt):.4f}")
print(f"Variance          : {float(x_opt @ toy_sigma @ x_opt):.4f}")


## 5. From QUBO to Ising Hamiltonian

Quantum hardware works in terms of Pauli operators, which have eigenvalues
$\pm 1$, not $0$ and $1$. The dictionary between binary $x_i$ and spin
$z_i$ is

$$
x_i \;=\; \frac{1 - z_i}{2}.
$$

So $z_i = +1$ (the $|0\rangle$ state) means $x_i = 0$ (stock not held),
and $z_i = -1$ (the $|1\rangle$ state) means $x_i = 1$ (stock held).

Substituting into $x^\top Q x$ and collecting terms gives the **Ising
form**:

$$
H \;=\; \sum_{i < j} J_{ij}\, Z_i Z_j \;+\; \sum_i h_i\, Z_i \;+\; \text{offset},
$$

where

$$
J_{ij} = \tfrac{Q_{ij}}{4}, \qquad
h_i = -\tfrac{1}{2} Q_{ii} - \tfrac{1}{4} \sum_{j \ne i} Q_{ij}.
$$

The ground state of $H$ — its lowest-eigenvalue eigenstate — is the
computational-basis state corresponding to the optimal portfolio.

Reference: **[3]** Lucas (2014).


In [ ]:
from src.qubo import qubo_to_ising, qubo_to_cudaq_hamiltonian

J, h, offset = qubo_to_ising(Q)
print(f"Number of Z_i Z_j coupling terms : {(J != 0).sum() // 2}")
print(f"Number of single-Z field terms   : {(h != 0).sum()}")
print(f"Energy offset (constant)         : {offset:.4f}")
print(f"\nh (single-qubit fields):\n{h}")


The Ising data above is what we'll build a `cudaq.SpinOperator` from. That
operator is what `cudaq.observe()` evaluates against the quantum state to
get the cost expectation value driving the optimiser.


In [ ]:
H_cudaq = qubo_to_cudaq_hamiltonian(Q)
print(f"cudaq SpinOperator built.")
print(f"  Number of terms: {H_cudaq.get_term_count()}")
print(f"\nFirst few terms:")
print(str(H_cudaq)[:400])


## 6. Stock Clustering — Splitting the Problem

A 15-qubit problem is fine on a single QPU, but 100 or 500 qubits is not.
The **divide-and-conquer** trick: cluster correlated stocks together,
solve each cluster independently on its own QPU, then merge.

The per-cluster QUBO drops the cross-cluster $\Sigma_{ij}$ entries — that
information is "lost" until the merge step recovers it via local search.
The smarter the clustering, the less information is lost.

### The distance formula

Correlation $\rho \in [-1, 1]$ is a similarity, not a distance. Convert
it via

$$
d_{ij} \;=\; \sqrt{2 (1 - \rho_{ij})}.
$$

This is a **proper metric** (it satisfies the triangle inequality), which
Ward's algorithm requires. Geometrically: if you treat each stock's
return series as a unit vector with dot product $\rho_{ij}$, then $d_{ij}$
is the Euclidean distance between the two unit vectors. So perfectly
correlated stocks ($\rho = 1$) are at distance $0$; uncorrelated stocks
($\rho = 0$) are at $\sqrt{2}$; perfectly anti-correlated stocks
($\rho = -1$) are at $2$.

Reference: **[6]** López de Prado (2016).

### Ward linkage

We use **agglomerative hierarchical clustering**: start with each stock
as its own cluster, then repeatedly merge the two clusters whose merger
increases total within-cluster variance the least. Cut the resulting tree
at $k = 4$ to get four clusters — one per QPU.


In [ ]:
from src.clustering import (
    cluster_stocks, build_subproblems, plot_dendrogram,
    plot_clustered_heatmap, compute_cross_cluster_loss,
)

N_CLUSTERS = 4
labels = cluster_stocks(rho, n_clusters=N_CLUSTERS)
print("Cluster assignment:")
for c in range(N_CLUSTERS):
    members = [TICKERS[i] for i in range(len(TICKERS)) if labels[i] == c]
    print(f"  Cluster {c}: {members}")


Visualise the merge order with a **dendrogram**. Vertical position of
each merge represents the Ward-linkage distance — the lower the merge,
the more similar the two sub-clusters being joined.


In [ ]:
plot_dendrogram(rho, TICKERS, save_path="dendrogram.png", n_clusters=N_CLUSTERS)
from IPython.display import Image
Image("dendrogram.png")


The same data can be visualised by reordering the correlation heatmap so
that members of each cluster are adjacent. Good clusters appear as bright
diagonal blocks separated by dim off-diagonal regions.


In [ ]:
plot_clustered_heatmap(rho, labels, TICKERS, save_path="clustered_heatmap.png")
Image("clustered_heatmap.png")


### Sub-budget allocation

Total budget is $B = 6$, but our 4 clusters have unequal sizes. Each
cluster gets a sub-budget proportional to its size, but **integer**.
That's not generally possible exactly, so we use the **largest-remainder
(Hamilton) method** — the same method used in political seat
apportionment:

1. Compute the ideal proportional allocation: $b_c^\star = B \cdot |C_c| / N$.
2. Floor each: $b_c = \lfloor b_c^\star \rfloor$.
3. Distribute the remaining seats $B - \sum_c b_c$ to the clusters with
   the largest fractional remainders.

`build_subproblems` does this and packages each cluster as a
self-contained mini-QUBO problem, complete with its sub-budget, local
$\mu$, local $\Sigma$, and the Ising terms ready for CUDA-Q.


In [ ]:
subproblems = build_subproblems(
    mu, sigma, labels,
    budget=BUDGET, q=Q_RISK, penalty=PENALTY, tickers=TICKERS,
)

for sp in subproblems:
    print(f"  Cluster {sp['cluster_id']:>1d} ({len(sp['stock_indices'])} stocks): "
          f"{sp['tickers']}, sub-budget = {sp['budget']}")


### Cross-cluster covariance loss

How much of the original problem are we throwing away by treating
clusters independently?

We compare the sum of squared covariances *between* clusters to the total
sum of squared covariances. That ratio is the fraction of "covariance
mass" our distributed solver is blind to. The local-search pass at the
end of the pipeline is what we use to recover it.


In [ ]:
loss = compute_cross_cluster_loss(sigma, labels)
print(f"Cross-cluster covariance loss: {loss * 100:.1f}%")


A loss in the $50$–$70\%$ range is high but normal for broadly correlated
equity markets where every stock shares "market beta". The merge +
local-search step in Section 9 recovers a meaningful fraction of it
without any extra quantum work.

Sources for the divide-and-conquer pattern: **[7]** NVIDIA cuda-q-academic
tutorials, **[9]** IonQ distributed portfolio optimization (2026).


## 7. QAOA — The Quantum Algorithm

### Intuition first, no jargon

1. Put all $2^n$ portfolios in **uniform superposition** — every possible
   selection is a candidate at once.
2. Apply a parameterised gate sequence that **stamps the energy** of each
   portfolio onto its quantum phase, then **interferes** the candidates
   so that low-energy states accumulate amplitude and high-energy states
   cancel.
3. **Measure**. Sample many times. The most common bitstring is (with
   high probability) a low-energy portfolio.

The trick: the parameterised gates have free parameters $(\gamma, \beta)$
that we tune with a **classical optimiser** to maximise the chance of
landing on a good portfolio.

### The circuit, layer by layer

Let $H_C$ be the cost Hamiltonian (the Ising form from Section 5) and
$H_M = \sum_i X_i$ be the **mixer**. For depth $p$, the circuit applies

$$
|\psi(\gamma, \beta)\rangle
\;=\;
\underbrace{e^{-i \beta_p H_M}\, e^{-i \gamma_p H_C}}_{\text{layer }p}
\cdots
\underbrace{e^{-i \beta_1 H_M}\, e^{-i \gamma_1 H_C}}_{\text{layer }1}
\, H^{\otimes n} |0\rangle^{\otimes n}.
$$

In gate primitives:

- **Initial state.** $H^{\otimes n}|0\rangle^{\otimes n}$ — Hadamards
  produce the uniform superposition over all $2^n$ portfolios.
- **Problem unitary $e^{-i \gamma H_C}$.** For each $J_{ij} Z_i Z_j$
  term we use the standard *CNOT — Rz($2\gamma J_{ij}$) — CNOT*
  decomposition. For each $h_i Z_i$ term we apply Rz($2\gamma h_i$) on
  qubit $i$.
- **Mixer $e^{-i \beta H_M}$.** Rx($2\beta$) on every qubit.

Total parameter count is $2p$ — a small number, even for many qubits.
That's what makes QAOA tractable for classical optimisation.

### The classical loop

Inner loop (per QAOA evaluation):
1. Pick parameters $(\gamma, \beta) \in \mathbb{R}^{2p}$.
2. Build and run the circuit; estimate
   $\langle \psi(\gamma, \beta) | H_C | \psi(\gamma, \beta) \rangle$
   via `cudaq.observe()`.

Outer loop (parameter optimisation):
- Use **COBYLA** (Constrained Optimization BY Linear Approximations) to
  minimise the expectation. COBYLA is gradient-free, builds a local
  linear model from a simplex of recent evaluations, and updates it
  trust-region-style.

Why COBYLA? It's the standard first choice for QAOA because:

- The energy landscape has many flat regions and barren plateaus where
  numerical gradients are unreliable.
- It has very few hyperparameters and converges in tens of iterations
  for small $p$.
- It's what **[2]** Buonaiuto et al. (2023), the NVIDIA CUDA-Q QAOA
  tutorials, and Qiskit's tutorials all use as their default.

References: **[5]** Powell (1994), **[4]** Farhi, Goldstone & Gutmann
(2014).

Run QAOA at depths $p = 1, 2, 3$ on the smallest cluster — small enough
to compare against an exact brute-force solution.


In [ ]:
from src.qaoa_cold import run_qaoa, evaluate_portfolio

small_sp = min(subproblems, key=lambda s: s["qubo"].shape[0])
print(f"Running QAOA on cluster {small_sp['cluster_id']}: "
      f"{small_sp['tickers']}, sub-budget = {small_sp['budget']}\n")

results_by_p = {}
for p in [1, 2, 3]:
    res = run_qaoa(
        small_sp["qubo"], small_sp["qubit_count"],
        layer_count=p, seed=42, shots=10000, maxiter=200,
    )
    results_by_p[p] = res
    print(f"  p={p}: best bitstring = {res['best_bitstring']}, "
          f"energy = {res['optimal_energy']:.4f}")


The convergence trace shows COBYLA's running cost expectation as it
adjusts $(\gamma, \beta)$ at each depth. Deeper circuits have more
parameters and more expressive states, but also a harder optimisation
landscape.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for p, res in results_by_p.items():
    if "convergence" in res and len(res["convergence"]) > 0:
        ax.plot(res["convergence"], label=f"p = {p}")
ax.set_xlabel("COBYLA iteration")
ax.set_ylabel(r"$\langle H_C \rangle$ (cost expectation)")
ax.set_title(f"QAOA convergence on cluster {small_sp['cluster_id']}")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


Compare each depth's QAOA solution to the exact brute-force optimum on
this cluster. The **approximation ratio (AR)** is the QAOA energy divided
by the brute-force optimum — values close to $1$ mean we've matched (or
nearly matched) the true minimum.


In [ ]:
bf_x, bf_val = brute_force_qubo(small_sp["qubo"], budget=small_sp["budget"])
bf_bs = "".join(str(int(b)) for b in bf_x)
print(f"Brute-force optimum: x = {bf_bs}  (energy = {bf_val:.4f})")
print(f"Selected: {[small_sp['tickers'][i] for i, b in enumerate(bf_x) if b==1]}\n")

print(f"{'p':>3s} {'QAOA bitstring':>16s} {'QAOA energy':>14s} {'AR':>10s}")
for p, res in results_by_p.items():
    ar = bf_val / res['optimal_energy'] if res['optimal_energy'] != 0 else float("nan")
    print(f"{p:>3d} {res['best_bitstring']:>16s} {res['optimal_energy']:>14.4f} {ar:>10.4f}")


## 8. Distributed Execution Across Multiple QPUs

The point of clustering was so that the per-cluster QAOA runs can happen
**in parallel**.

```
sequential (1 QPU):    [Cluster 0]→[Cluster 1]→[Cluster 2]→[Cluster 3]
                        |--------------------------------------------|
                                          total time

distributed (4 QPUs):  [Cluster 0]   on QPU 0
                       [Cluster 1]   on QPU 1     all simultaneous
                       [Cluster 2]   on QPU 2
                       [Cluster 3]   on QPU 3
                        |-------|
                          ~ max time
```

### CUDA-Q MQPU backend

NVIDIA CUDA-Q exposes a *multi-processor platform* abstraction where each
visible GPU appears as a virtual QPU. The relevant API:

- `cudaq.set_target("nvidia", option="mqpu")` — switch to the multi-QPU
  backend; one virtual QPU per GPU.
- `cudaq.sample_async(kernel, ..., qpu_id=i)` — non-blocking dispatch to
  QPU `i`. Returns an `AsyncSampleResult` future.
- `cudaq.observe_async(kernel, hamiltonian, ..., qpu_id=i)` — same idea
  for expectation values.
- `future.get()` — block until the assigned QPU returns the result.

We use **round-robin scheduling**: cluster $c$ goes to QPU
$c \bmod n_{\text{QPUs}}$.

Reference: **[8]** NVIDIA CUDA-Q *Multi-Processor Platforms* documentation.

First, configure MQPU and report what we have available.


In [ ]:
from src.distributed import setup_mqpu, run_distributed_cold_qaoa

info = setup_mqpu()
print(f"Backend     : {info['backend']}")
print(f"Num QPUs    : {info['num_qpus']}")
print(f"Note        : {info['note']}")


Now dispatch the four cluster QAOAs across the available QPUs. With the
`nvidia,mqpu` backend each cluster runs on its own GPU; the function also
falls back to sequential execution on `qpp-cpu` or numpy if no GPUs are
visible.


In [ ]:
distributed_results = run_distributed_cold_qaoa(
    subproblems, layer_count=2, seed=42, shots=10000, maxiter=200,
)
print(f"Got {len(distributed_results)} per-cluster results.")


Per-cluster summary: which QPU handled which cluster, what bitstring the
sampling produced, and the energy of that bitstring under the cluster's
local QUBO.


In [ ]:
print(f"{'Cluster':<8s} {'Tickers':<32s} {'Bitstring':<14s} "
      f"{'Sampled E':>12s} {'Optimal E':>12s} {'QPU':>4s} {'Time (s)':>10s}")
print("-" * 100)
for r in distributed_results:
    sp = next(s for s in subproblems if s["cluster_id"] == r["cluster_id"])
    tickers_str = ",".join(sp["tickers"])
    print(f"{r['cluster_id']:<8d} {tickers_str:<32s} {r['best_bitstring']:<14s} "
          f"{r['sampled_energy']:>12.4f} {r['optimal_energy']:>12.4f} "
          f"{r['qpu_id']:>4d} {r['timing_s']:>10.2f}")


## 9. Merging and Post-Processing

### The index mapping problem

Each cluster gives us a **local** bitstring of length $|C_c|$ — the
selection within that cluster's own indexing. To get a single global
portfolio of length $N=15$ we have to map those local indices back to
their original positions in the universe.

Concretely, if cluster 0 contains stocks at global indices `[6, 7, 8]`
and the QAOA returned `"010"`, then globally we have $x_6 = 0$,
$x_7 = 1$, $x_8 = 0$. `merge_subportfolios` does exactly this rebinding.


In [ ]:
from src.merge import merge_subportfolios, evaluate_full_portfolio, local_search

x_merged = merge_subportfolios(distributed_results, subproblems, n_total=len(TICKERS))
print(f"Merged binary vector x:\n  {x_merged.astype(int)}")
print(f"\nSelected stocks ({int(x_merged.sum())} chosen):")
for i, b in enumerate(x_merged):
    if b == 1:
        print(f"  {TICKERS[i]}")


Now evaluate the merged portfolio using the **full** $\mu$ and $\Sigma$
— including the cross-cluster covariance entries that the distributed
solve ignored. Expect risk to look worse here than the per-cluster
energies suggested, because those local energies didn't see cross-cluster
covariance.


In [ ]:
port_merged = evaluate_full_portfolio(x_merged, mu, sigma, q=Q_RISK)

print("Merged portfolio (before local search):")
print(f"  Stocks selected : {[TICKERS[i] for i in port_merged['selected_indices']]}")
print(f"  n_selected      : {port_merged['n_selected']}  (target = {BUDGET})")
print(f"  Expected return : {port_merged['expected_return']:+.4f}")
print(f"  Risk (sigma)    : {port_merged['risk']:.4f}")
print(f"  Sharpe ratio    : {port_merged['sharpe_ratio']:.4f}")
print(f"  Markowitz obj   : {port_merged['markowitz_obj']:.4f}")


### Local search

The merged portfolio ignored a large fraction of the covariance
structure (the cross-cluster percentage we computed in Section 6). A
cheap way to recover some of that is **greedy 1-swap local search** on
the *full* QUBO matrix:

1. Look at the currently selected stocks $S$ and unselected stocks
   $\bar S$.
2. For every pair $(i \in S,\, j \in \bar S)$ try swapping: remove
   $i$, add $j$. Compute $\Delta = (x')^\top Q x' - x^\top Q x$.
3. If any swap has $\Delta < 0$, take the *best* such swap and repeat.
   If no improving swap exists, stop.

This is **budget-preserving by construction** — we always remove one and
add one. And it always evaluates against the full $15 \times 15$ QUBO,
so it directly fights the cross-cluster information loss without any
extra quantum work.


In [ ]:
Q_global = build_qubo(mu, sigma, q=Q_RISK, budget=BUDGET, penalty=PENALTY)
x_polished = local_search(x_merged, Q_global, budget=BUDGET, max_swaps=100)
n_swaps = int((x_polished != x_merged).sum() // 2)
print(f"Local search made {n_swaps} swap(s).")


Evaluate the polished portfolio against the full covariance matrix and
compare numerically against the unpolished merge.


In [ ]:
port_polished = evaluate_full_portfolio(x_polished, mu, sigma, q=Q_RISK)

print("After greedy local search:")
print(f"  Stocks selected : {[TICKERS[i] for i in port_polished['selected_indices']]}")
print(f"  n_selected      : {port_polished['n_selected']}  (target = {BUDGET})")
print(f"  Expected return : {port_polished['expected_return']:+.4f}")
print(f"  Risk (sigma)    : {port_polished['risk']:.4f}")
print(f"  Sharpe ratio    : {port_polished['sharpe_ratio']:.4f}")
print(f"  Markowitz obj   : {port_polished['markowitz_obj']:.4f}")


## 10. Results and Analysis

We can compare three portfolios on the **full** problem:

1. **Brute force** — enumerate all $\binom{15}{6} = 5005$ feasible
   portfolios and pick the one with the lowest QUBO energy. Ground truth
   for our parameter choice $(B, q, \lambda)$.
2. **Distributed cold-start QAOA (merged)** — the un-polished output of
   the divide-and-conquer pipeline.
3. **Distributed QAOA + local search** — same, after greedy 1-swap
   refinement.

The **approximation ratio** is defined as the ratio of QUBO energies
relative to brute force. Because all three energies are negative
(infeasible portfolios are pushed up by the $+\lambda B^2$ penalty), AR
is just $E_{\text{method}} / E_{\text{brute force}}$ — values near $1$
mean we've matched the optimum.


In [ ]:
bf_x_global, bf_val_global = brute_force_qubo(Q_global, budget=BUDGET)
port_bf = evaluate_full_portfolio(bf_x_global, mu, sigma, q=Q_RISK)

E_bf       = bf_val_global
E_merged   = float(x_merged   @ Q_global @ x_merged)
E_polished = float(x_polished @ Q_global @ x_polished)

def AR(E):
    return E / E_bf if abs(E_bf) > 1e-12 else float("nan")

rows = [
    ("Brute force (global)",        port_bf,       E_bf,       1.0),
    ("Distributed QAOA (merged)",   port_merged,   E_merged,   AR(E_merged)),
    ("Distributed QAOA + local",    port_polished, E_polished, AR(E_polished)),
]

print(f"{'Method':<28s} {'Stocks':<32s} {'QUBO E':>10s} {'AR':>8s} "
      f"{'Return':>8s} {'Risk':>8s} {'Sharpe':>8s}")
print("-" * 110)
for name, port, E, ar in rows:
    stocks = ",".join(TICKERS[i] for i in port["selected_indices"])
    print(f"{name:<28s} {stocks:<32s} {E:>10.2f} {ar:>8.4f} "
          f"{port['expected_return']:>8.4f} {port['risk']:>8.4f} "
          f"{port['sharpe_ratio']:>8.4f}")


### Approximation ratios as a bar chart

A side-by-side picture of how close each method gets to the brute-force
ground truth.


In [ ]:
labels_bar = ["Brute force", "QAOA (merged)", "QAOA + local"]
ars        = [1.0, AR(E_merged), AR(E_polished)]
colors_bar = ["#444444", "#3b76af", "#519e3e"]

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(labels_bar, ars, color=colors_bar)
ax.axhline(1.0, ls="--", color="gray", lw=1, label="brute-force optimum")
ax.set_ylabel("approximation ratio  (E_method / E_brute-force)")
ax.set_ylim(0, max(1.05, max(ars) * 1.05))
ax.set_title("Approximation ratio against the brute-force optimum")
ax.legend()
for b, ar in zip(bars, ars):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01,
            f"{ar:.3f}", ha="center", va="bottom", fontsize=10)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


### Selected stocks visualisation

Which stocks did each method actually pick? Rows are methods, columns are
tickers; a filled cell means the stock is in the portfolio. This is the
clearest way to *see* whether local search and brute force agree.


In [ ]:
import matplotlib.colors as mcolors

methods = ["Brute force", "QAOA (merged)", "QAOA + local"]
xs = np.vstack([bf_x_global, x_merged, x_polished]).astype(int)

# Color each ticker column by its sector for quick orientation.
sector_of = {
    "AAPL": "Tech",  "MSFT": "Tech",  "GOOGL": "Tech",
    "JPM":  "Fin",   "GS":   "Fin",   "BAC":   "Fin",
    "XOM":  "Energy","CVX":  "Energy","COP":   "Energy",
    "JNJ":  "Health","PFE":  "Health","UNH":   "Health",
    "PG":   "Cons",  "KO":   "Cons",  "WMT":   "Cons",
}
sector_colors = {
    "Tech":   "#3b76af",
    "Fin":    "#e7702a",
    "Energy": "#519e3e",
    "Health": "#c43d3d",
    "Cons":   "#8b6caf",
}

fig, ax = plt.subplots(figsize=(11, 3.2))
for i, t in enumerate(TICKERS):
    for j in range(len(methods)):
        if xs[j, i]:
            ax.add_patch(plt.Rectangle((i, j), 1, 1,
                                       color=sector_colors[sector_of[t]]))
        ax.add_patch(plt.Rectangle((i, j), 1, 1,
                                   fill=False, edgecolor="white", lw=1))
ax.set_xlim(0, len(TICKERS))
ax.set_ylim(0, len(methods))
ax.set_xticks(np.arange(len(TICKERS)) + 0.5)
ax.set_xticklabels(TICKERS, rotation=45, ha="right")
ax.set_yticks(np.arange(len(methods)) + 0.5)
ax.set_yticklabels(methods)
ax.set_title("Stock selections by method (cell coloured by sector)")
ax.invert_yaxis()
# Sector legend
handles = [plt.Rectangle((0,0), 1, 1, color=c, label=s)
           for s, c in sector_colors.items()]
ax.legend(handles=handles, loc="upper center", ncol=5,
          bbox_to_anchor=(0.5, -0.25))
plt.tight_layout()
plt.show()


### Discussion

**How close did distributed QAOA get to the global optimum?**

The cold-start QAOA, run independently per cluster, has each sub-solve
seeing only its local QUBO. Its merged solution gets an approximation
ratio (relative to global brute force) that depends heavily on:

- the cross-cluster covariance loss (a structural property of how
  correlated the universe is — see Section 6);
- the per-cluster QAOA quality at depth $p=2$ with COBYLA's 200 iteration
  budget;
- the random parameter seeds.

Typical AR values for this universe are in the $0.7$–$1.0$ range for the
merged portfolio.

**Did local search help?**

Local search only sees improvements: by construction it never makes the
QUBO objective worse. It can only do nothing or strictly improve. In
practice, on a 15-stock universe with $\sim 60\%$ cross-cluster loss, a
single 1-swap pass typically lifts AR by $0.1$–$0.3$, often closing the
gap to the brute-force optimum entirely.

**What was the cost of distributing?**

The distributed solver was *blind* to the cross-cluster covariance — that
$\sim 60\%$ of the covariance mass we measured in Section 6. Local search
recovers most of it classically; what's left is the genuine "cost" of
divide-and-conquer at this universe size. For larger universes where
brute force is infeasible, this cost is the price paid for the
parallelism that lets quantum solvers scale at all.


## 11. References

1. **H. Markowitz**, *"Portfolio Selection,"* **Journal of Finance** 7(1),
   77–91 (1952). — The original mean-variance framework.
2. **F. Buonaiuto, F. Gargiulo, G. De Pietro, M. Esposito, M. Pota**,
   *"Best practices for portfolio optimization by quantum computing,
   experimented on real quantum devices,"* **Scientific Reports** 13,
   19434 (2023). — QUBO formulation and penalty design used here.
3. **A. Lucas**, *"Ising formulations of many NP problems,"*
   **Frontiers in Physics** 2:5 (2014).
   [arXiv:1302.5843](https://arxiv.org/abs/1302.5843).
   — QUBO ↔ Ising mapping.
4. **E. Farhi, J. Goldstone, S. Gutmann**, *"A Quantum Approximate
   Optimization Algorithm,"* (2014).
   [arXiv:1411.4028](https://arxiv.org/abs/1411.4028). — The original
   QAOA paper.
5. **M. J. D. Powell**, *"A Direct Search Optimization Method that
   Models the Objective and Constraint Functions by Linear
   Interpolation,"* in **Advances in Optimization and Numerical
   Analysis**, Kluwer Academic, 1994. — The COBYLA optimiser.
6. **M. López de Prado**, *"Building Diversified Portfolios that
   Outperform Out of Sample,"* **Journal of Portfolio Management**
   42(4), 59–69 (2016). — Correlation-distance metric and hierarchical
   risk parity.
7. **NVIDIA**, *cuda-q-academic*, GitHub repository.
   [github.com/NVIDIA/cuda-q-academic](https://github.com/NVIDIA/cuda-q-academic).
   — Distributed Max-Cut and clustering tutorials with the same
   divide-and-conquer pattern used in this notebook.
8. **NVIDIA**, *CUDA-Q Multi-Processor Platforms documentation*.
   [nvidia.github.io/cuda-quantum](https://nvidia.github.io/cuda-quantum/).
   — `set_target("nvidia", option="mqpu")`, `sample_async`,
   `observe_async`, and `qpu_id`-based dispatch.
9. **IonQ**, *Distributed Portfolio Optimization*, white paper / preprint
   (2026). — Inspiration for the divide-and-conquer + merge pattern at
   the application layer.
10. **D. J. Egger, J. Mareček, S. Woerner**, *"Warm-starting quantum
    optimization,"* **Quantum** 5, 479 (2021).
    [arXiv:2009.10095](https://arxiv.org/abs/2009.10095). — Warm-start
    QAOA via Goemans–Williamson SDP rounding; mentioned as future work.
